# Stage 3 — Fine-tuning SHViT-S4 on Food-101

This notebook fine-tunes a pretrained SHViT model on Food-101 using `finetune_shvit_food101.py`.

**Before running:** `Runtime → Change runtime type → T4 GPU`

### What this notebook does
1. Verifies GPU, clones repos, installs deps
2. Downloads SHViT-S4 pretrained weights
3. Runs the fine-tuning script for 30 epochs
4. Plots training curves from the CSV log

### Key design decisions (matching original SHViT paper)
| Setting | Value | Source |
|---|---|---|
| Augmentation | RandAugment `rand-m9-mstd0.5-inc1` + RandomErasing p=0.25 | SHViT paper |
| Mixup | alpha=0.8 | SHViT paper |
| CutMix | alpha=1.0 | SHViT paper |
| Label smoothing | 0.1 | SHViT paper |
| Training FP | **Full FP32** forward pass | Matches engine.py (commented-out autocast) |
| Eval FP | AMP autocast | Matches engine.py `evaluate()` |
| Grad clip | norm 0.02 | SHViT paper |
| Optimizer | AdamW, wd=0.025 | SHViT paper |
| LR schedule | Cosine + 5-epoch warmup | SHViT paper |

## 0. GPU check

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

## 1. (Optional) Mount Drive

Persist the dataset, weights, and checkpoints across sessions.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT   = '/content/drive/MyDrive/food101_data'
    WEIGHTS_DIR = '/content/drive/MyDrive/shvit_weights'
    OUTPUT_DIR  = '/content/drive/MyDrive/shvit_finetune'
else:
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/weights'
    OUTPUT_DIR  = '/content/checkpoints/shvit_food101'

WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'
print('Data      :', DATA_ROOT)
print('Weights   :', WEIGHTS_PATH)
print('Output    :', OUTPUT_DIR)

## 2. Clone repos and install dependencies

In [ ]:
import os

# Clone SHViT (provides the model architecture)
if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT

# Clone our project repo to get the fine-tuning script and helpers
if not os.path.isdir('/content/Vision_Project_spring_26'):
    !git clone -b claude/setup-shvit-repo-6OXc1 \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git \
        /content/Vision_Project_spring_26

import shutil
REPO = '/content/Vision_Project_spring_26'
for fname in [
    'Stage 3: fine-tuning SHViT/finetune_shvit_food101.py',
    'splits.py',
    'metrics.py',
    'train_val_split_seed42.json',
]:
    shutil.copy(f'{REPO}/{fname}', f'/content/{os.path.basename(fname)}')
    print('Copied:', os.path.basename(fname))


In [ ]:
# Install SHViT deps. timm must be pinned to 0.5.4 and installed --no-deps
# to avoid downgrading Colab's PyTorch. scikit-image / fvcore / yacs / onnx
# from SHViT's requirements.txt are NOT needed by our fine-tuning wrapper —
# scikit-image==0.19.3 in particular has no wheels for Python 3.12 and would
# fail to build from source.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict
print('Dependencies installed.')

## 3. Download SHViT-S4 pretrained weights

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)
if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

## 4. Fine-tune SHViT-S4 on Food-101

Food-101 is ~5 GB and downloaded automatically on the first run.

### Estimated runtime on T4
Each epoch: ~5-7 min (75k train images, batch 64, full FP32 forward).  
30 epochs ≈ **2.5–3.5 hours** — use `--epochs 5` first to smoke-test.

> **Tip:** Mount Drive (Cell 1) so checkpoints survive session expiry.

In [ ]:
# Quick smoke-test: 2 epochs to verify everything runs
!python /content/finetune_shvit_food101.py \
    --shvit-dir  /content/SHViT \
    --finetune   {WEIGHTS_PATH} \
    --data-root  {DATA_ROOT} \
    --output-dir {OUTPUT_DIR} \
    --split-file /content/train_val_split_seed42.json \
    --epochs 2 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 2

In [ ]:
# Full 30-epoch run — comment out the smoke-test cell above first
!python /content/finetune_shvit_food101.py \
    --shvit-dir  /content/SHViT \
    --finetune   {WEIGHTS_PATH} \
    --data-root  {DATA_ROOT} \
    --output-dir {OUTPUT_DIR} \
    --split-file /content/train_val_split_seed42.json \
    --epochs 30 \
    --batch-size 64 \
    --lr 1e-4 \
    --warmup-epochs 5 \
    --weight-decay 0.025 \
    --clip-grad 0.02 \
    --mixup 0.8 \
    --cutmix 1.0 \
    --smoothing 0.1 \
    --aa rand-m9-mstd0.5-inc1 \
    --reprob 0.25 \
    --save-freq 10 \
    --num-workers 2

In [ ]:
# Resume from a checkpoint if the session was interrupted
# !python /content/finetune_shvit_food101.py \
#     --shvit-dir  /content/SHViT \
#     --data-root  {DATA_ROOT} \
#     --output-dir {OUTPUT_DIR} \
#     --resume     {OUTPUT_DIR}/checkpoint_010.pth \
#     --epochs 30 \
#     --batch-size 64 \
#     --lr 1e-4 \
#     --num-workers 2

## 5. Evaluate the best checkpoint

In [ ]:
!python /content/finetune_shvit_food101.py \
    --shvit-dir  /content/SHViT \
    --finetune   {OUTPUT_DIR}/best.pth \
    --data-root  {DATA_ROOT} \
    --output-dir {OUTPUT_DIR} \
    --split-file /content/train_val_split_seed42.json \
    --eval

## 6. Plot training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'{OUTPUT_DIR}/training_log.csv'
df = pd.read_csv(csv_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].plot(df['epoch'], df['val_loss'],   label='val', linestyle='--')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch')
axes[0].legend(); axes[0].grid(True)

# Accuracy
axes[1].plot(df['epoch'], df['val_top1'] * 100, label='top-1')
axes[1].plot(df['epoch'], df['val_top5'] * 100, label='top-5', linestyle='--')
axes[1].set_title('Val Accuracy (%)'); axes[1].set_xlabel('epoch')
axes[1].legend(); axes[1].grid(True)

# LR schedule
axes[2].plot(df['epoch'], df['lr'].astype(float))
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('epoch')
axes[2].set_yscale('log'); axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120)
plt.show()

best = df.loc[df['val_top1'].idxmax()]
print(f"Best  top-1: {best['val_top1']*100:.2f}%  "
      f"top-5: {best['val_top5']*100:.2f}%  "
      f"(epoch {int(best['epoch'])})")